# MODULE 7 — Classical Baselines and EEGNet Reference

**Project:** Strict Cross-Dataset, Subject-Independent Motor-Imagery EEG Classification

## Objective

Establish reproducible reference performance before implementing the proposed
domain-generalization architecture.

The same Module 6 cache and fold manifests are used for every baseline.

## Baselines

### B1 — CSP + LDA
One-versus-rest multiclass spatial filtering followed by log-variance features
and Linear Discriminant Analysis.

### B2 — FBCSP + LDA
Five filter-bank bands within the frozen 8–30 Hz representation:

- 8–12 Hz
- 12–16 Hz
- 16–20 Hz
- 20–24 Hz
- 24–30 Hz

CSP is independently fitted within each source-training fold and the resulting
log-variance features are concatenated before LDA.

### B3 — Log-Euclidean covariance + LDA
A compact Riemannian-style covariance baseline:
- shrinkage covariance per epoch;
- matrix logarithm;
- symmetric-matrix vectorization;
- LDA.

The transform is fitted only from source-training epochs.

### B4 — EEGNet reference
A compact EEGNet-style neural baseline using the same 22×640 input.

The EEGNet normalizer is fitted only on source training subjects.

## Evaluation protocols

### Within-dataset LOSO
- BCI-IV-2a: 9 target subjects
- EEGMMIDB: 109 target subjects

### Cross-dataset zero-calibration
- BCI-IV-2a → EEGMMIDB: 109 target subjects
- EEGMMIDB → BCI-IV-2a: 9 target subjects

No target data are used for:
- model fitting;
- feature fitting;
- normalization;
- CSP fitting;
- covariance reference fitting;
- hyperparameter selection.

## Metrics

Primary:
- accuracy
- balanced accuracy
- macro F1

Secondary:
- per-class recall
- confusion matrix
- fold-level mean ± standard deviation

For the 3-class task, chance level is 33.33%.

## Compute policy

The notebook supports:
- fast smoke tests;
- resumable full baseline runs;
- MPS on Apple Silicon when available;
- CPU fallback.

Classical baselines are relatively inexpensive.

EEGNet can be substantially more expensive over 118 LOSO folds, so it is
controlled by explicit configuration flags and writes results after every fold.

## Cell 1 — Imports, configuration, device

In [1]:
# ============================================================
# CELL 1 — IMPORTS + CONFIGURATION
# ============================================================

from __future__ import annotations

import os
import json
import time
import random
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import h5py

from scipy import signal
from scipy.linalg import eigh

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
)

from sklearn.covariance import LedoitWolf

import mne
from mne.decoding import CSP

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/Users/ashokvarmabevara/Project2"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "cross_dataset_mi_project"
)

MANIFEST_ROOT = (
    OUTPUT_ROOT
    / "manifests"
)

CACHE_ROOT = (
    OUTPUT_ROOT
    / "cache"
)

RESULTS_ROOT = (
    OUTPUT_ROOT
    / "results"
)

BASELINE_ROOT = (
    RESULTS_ROOT
    / "module_7_baselines"
)

BASELINE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

CACHE_PATH = (
    CACHE_ROOT
    / "module_5_v2_preprocessed_epochs_160hz_8_30hz_continuous.h5"
)

WITHIN_LOSO_PATH = (
    MANIFEST_ROOT
    / "module_6_within_dataset_loso_folds.csv"
)

TRANSFER_PATH = (
    MANIFEST_ROOT
    / "module_6_cross_dataset_transfer_folds.csv"
)

CACHE_META_PATH = (
    MANIFEST_ROOT
    / "module_6_cache_metadata.csv"
)

PROTOCOL_PATH = (
    MANIFEST_ROOT
    / "module_6_baseline_protocol.json"
)

for path in [
    CACHE_PATH,
    WITHIN_LOSO_PATH,
    TRANSFER_PATH,
    CACHE_META_PATH,
    PROTOCOL_PATH,
]:
    assert path.exists(), f"Missing artifact: {path}"

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 20260822

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print("=" * 78)
print("MODULE 7 — BASELINE EXPERIMENTS")
print("=" * 78)

print("PyTorch:", torch.__version__)
print("Device :", DEVICE)
print("Seed   :", SEED)

MODULE 7 — BASELINE EXPERIMENTS
PyTorch: 2.10.0
Device : mps
Seed   : 20260822


## Cell 2 — Frozen experiment configuration

In [2]:
# ============================================================
# CELL 2 — FROZEN BASELINE CONFIGURATION
# ============================================================

PRIMARY_CLASSES = [
    "left",
    "right",
    "feet",
]

CLASS_TO_ID = {
    "left": 0,
    "right": 1,
    "feet": 2,
}

ID_TO_CLASS = {
    v: k
    for k, v in CLASS_TO_ID.items()
}

# ------------------------------------------------------------
# Classical baselines
# ------------------------------------------------------------

CSP_N_COMPONENTS = 6

FBCSP_BANDS = [
    (8.0, 12.0),
    (12.0, 16.0),
    (16.0, 20.0),
    (20.0, 24.0),
    (24.0, 30.0),
]

FBCSP_N_COMPONENTS = 4

# Riemannian covariance
RIEMANNIAN_REG = "oas"

# ------------------------------------------------------------
# EEGNet reference
# ------------------------------------------------------------

EEGNET_EPOCHS = 30
EEGNET_BATCH_SIZE = 128
EEGNET_LR = 1e-3
EEGNET_WEIGHT_DECAY = 1e-4
EEGNET_PATIENCE = 6

# ------------------------------------------------------------
# Run controls
#
# Classical baselines are enabled.
# EEGNet is enabled but can be disabled for a quick pass.
# ------------------------------------------------------------

RUN_CSP = True
RUN_FBCSP = True
RUN_RIEMANNIAN = True
RUN_EEGNET = True

# Set this to True for one-fold smoke validation.
SMOKE_TEST = False

# If SMOKE_TEST=False, all LOSO/cross-dataset folds are run.
SMOKE_FOLDS_PER_PROTOCOL = 1

# EEGNet can be computationally expensive across 118 folds.
EEGNET_FULL_WITHIN_LOSO = True
EEGNET_FULL_CROSS_DATASET = True

# Classical results are written incrementally.
RESULTS_FLUSH_EVERY_FOLD = True

print(
    json.dumps(
        {
            "classes": PRIMARY_CLASSES,
            "csp_components": CSP_N_COMPONENTS,
            "fbcsp_bands": FBCSP_BANDS,
            "fbcsp_components": FBCSP_N_COMPONENTS,
            "riemannian_regularization": RIEMANNIAN_REG,
            "eegnet_epochs": EEGNET_EPOCHS,
            "eegnet_batch_size": EEGNET_BATCH_SIZE,
            "run_csp": RUN_CSP,
            "run_fbcsp": RUN_FBCSP,
            "run_riemannian": RUN_RIEMANNIAN,
            "run_eegnet": RUN_EEGNET,
            "smoke_test": SMOKE_TEST,
        },
        indent=2,
    )
)

{
  "classes": [
    "left",
    "right",
    "feet"
  ],
  "csp_components": 6,
  "fbcsp_bands": [
    [
      8.0,
      12.0
    ],
    [
      12.0,
      16.0
    ],
    [
      16.0,
      20.0
    ],
    [
      20.0,
      24.0
    ],
    [
      24.0,
      30.0
    ]
  ],
  "fbcsp_components": 4,
  "riemannian_regularization": "oas",
  "eegnet_epochs": 30,
  "eegnet_batch_size": 128,
  "run_csp": true,
  "run_fbcsp": true,
  "run_riemannian": true,
  "run_eegnet": true,
  "smoke_test": false
}


## Cell 3 — Load cache metadata and fold manifests

In [3]:
# ============================================================
# CELL 3 — LOAD METADATA + FOLD MANIFESTS
# ============================================================

cache_meta_df = pd.read_csv(
    CACHE_META_PATH
)

within_loso_df = pd.read_csv(
    WITHIN_LOSO_PATH
)

transfer_df = pd.read_csv(
    TRANSFER_PATH
)

print("Cache epochs :", len(cache_meta_df))
print(
    "Cache subjects:",
    cache_meta_df["subject"].nunique(),
)
print(
    "Within LOSO folds:",
    len(within_loso_df),
)
print(
    "Transfer folds:",
    len(transfer_df),
)

assert len(cache_meta_df) == 9316
assert cache_meta_df["subject"].nunique() == 118

assert len(within_loso_df) == 118
assert len(transfer_df) == 118

print("\nModule 6 artifacts loaded: PASS")

Cache epochs : 9316
Cache subjects: 118
Within LOSO folds: 118
Transfer folds: 118

Module 6 artifacts loaded: PASS


## Cell 4 — HDF5 batch reader

In [4]:
# ============================================================
# CELL 4 — HDF5 BATCH READER
# ============================================================

class HDF5Store:
    def __init__(self, path):
        self.path = Path(path)
        self.h5 = None

    def __enter__(self):
        self.h5 = h5py.File(
            self.path,
            "r",
        )
        return self

    def __exit__(
        self,
        exc_type,
        exc,
        tb,
    ):
        if self.h5 is not None:
            self.h5.close()
            self.h5 = None

    def get_X(
        self,
        indices,
    ):
        indices = np.asarray(
            indices,
            dtype=np.int64,
        )

        if len(indices) == 0:
            return np.empty(
                (0, 22, 640),
                dtype=np.float32,
            )

        order = np.argsort(
            indices
        )

        sorted_idx = indices[
            order
        ]

        X_sorted = np.asarray(
            self.h5["X"][
                sorted_idx
            ],
            dtype=np.float32,
        )

        inverse = np.argsort(
            order
        )

        return X_sorted[
            inverse
        ]

    def get_meta(
        self,
        indices,
    ):
        indices = np.asarray(
            indices,
            dtype=np.int64,
        )

        order = np.argsort(
            indices
        )

        sorted_idx = indices[
            order
        ]

        inverse = np.argsort(
            order
        )

        result = {}

        string_keys = [
            "dataset",
            "subject",
            "run",
            "recording_id",
            "harmonized_class",
        ]

        for key in string_keys:
            vals = self.h5[
                "metadata"
            ][key][
                sorted_idx
            ]

            result[key] = [
                x.decode("utf-8")
                if isinstance(x, bytes)
                else str(x)
                for x in vals
            ]

            result[key] = [
                result[key][i]
                for i in inverse
            ]

        result["event_index"] = (
            self.h5[
                "metadata"
            ]["event_index"][
                sorted_idx
            ][inverse]
        )

        result["source_sfreq_hz"] = (
            self.h5[
                "metadata"
            ]["source_sfreq_hz"][
                sorted_idx
            ][inverse]
        )

        return pd.DataFrame(
            result,
            index=indices,
        )


print("HDF5 reader ready.")

HDF5 reader ready.


## Cell 5 — Metrics and result schema

In [5]:
# ============================================================
# CELL 5 — METRICS
# ============================================================

def compute_metrics(
    y_true,
    y_pred,
):
    y_true = np.asarray(
        y_true,
        dtype=np.int64,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=np.int64,
    )

    return {
        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                average="macro",
            )
        ),
        "recall_left": float(
            f1_score(
                y_true,
                y_pred,
                labels=[0],
                average=None,
                zero_division=0,
            )[0]
        ),
        "recall_right": float(
            f1_score(
                y_true,
                y_pred,
                labels=[1],
                average=None,
                zero_division=0,
            )[0]
        ),
        "recall_feet": float(
            f1_score(
                y_true,
                y_pred,
                labels=[2],
                average=None,
                zero_division=0,
            )[0]
        ),
    }


def safe_protocol_indices(
    json_string,
):
    return np.asarray(
        json.loads(json_string),
        dtype=np.int64,
    )


def protocol_name_from_row(row):
    return str(
        row["protocol"]
    )


print(
    "Metrics and fold-index utilities: PASS"
)

Metrics and fold-index utilities: PASS


## Cell 6 — Source-only robust normalizer

In [6]:
# ============================================================
# CELL 6 — SOURCE-ONLY NORMALIZATION
# ============================================================

class FoldRobustNormalizer:
    """
    Fold-local channel-wise robust normalizer.

    Fits only on the source-training partition.
    """

    def __init__(
        self,
        eps=1e-6,
    ):
        self.eps = float(eps)
        self.median_ = None
        self.iqr_ = None
        self.fit_subjects_ = None

    def fit(
        self,
        X,
        subjects,
    ):

        X = np.asarray(
            X,
            dtype=np.float64,
        )

        subjects = [
            str(s)
            for s in subjects
        ]

        if X.ndim != 3:
            raise ValueError(X.shape)

        if len(subjects) != X.shape[0]:
            raise ValueError(
                "Subject count must match X epochs."
            )

        values = (
            X.transpose(
                1,
                0,
                2,
            )
            .reshape(
                22,
                -1,
            )
        )

        self.median_ = np.median(
            values,
            axis=1,
        )

        q25 = np.percentile(
            values,
            25,
            axis=1,
        )

        q75 = np.percentile(
            values,
            75,
            axis=1,
        )

        self.iqr_ = np.maximum(
            q75 - q25,
            self.eps,
        )

        self.fit_subjects_ = tuple(
            sorted(
                set(subjects)
            )
        )

        return self

    def transform(
        self,
        X,
    ):

        X = np.asarray(
            X,
            dtype=np.float64,
        )

        return (
            X
            - self.median_[
                None,
                :,
                None,
            ]
        ) / (
            self.iqr_[
                None,
                :,
                None,
            ]
        )

    def assert_target_excluded(
        self,
        target_subject,
    ):

        target_subject = str(
            target_subject
        )

        assert (
            target_subject
            not in set(
                self.fit_subjects_
            )
        ), (
            f"LEAKAGE: {target_subject} "
            "was used in normalizer fit."
        )


print(
    "Fold-local normalization utility: PASS"
)

Fold-local normalization utility: PASS


## Cell 7 — CSP + LDA implementation

CSP is fitted independently inside each fold.

The target subject is never involved in CSP fitting.

MNE's multiclass CSP generates a spatial representation followed by LDA.

In [7]:
# ============================================================
# CELL 7 — CSP + LDA
# ============================================================

def fit_csp_lda(
    X_train,
    y_train,
    n_components=CSP_N_COMPONENTS,
):

    csp = CSP(
        n_components=n_components,
        reg="oas",
        log=True,
        norm_trace=False,
        transform_into="average_power",
    )

    X_train_features = csp.fit_transform(
        X_train,
        y_train,
    )

    lda = LinearDiscriminantAnalysis(
        solver="lsqr",
        shrinkage="auto",
    )

    lda.fit(
        X_train_features,
        y_train,
    )

    return csp, lda


def predict_csp_lda(
    csp,
    lda,
    X_test,
):

    X_features = csp.transform(
        X_test
    )

    return lda.predict(
        X_features
    )


print("CSP + LDA implementation: READY")

CSP + LDA implementation: READY


## Cell 8 — FBCSP + LDA implementation

The 8–30 Hz cache is passed through a deterministic filter bank.

Each band gets:
- band-pass filter;
- CSP fit on source training data;
- log-power features.

Only source training data determine CSP spatial filters and LDA parameters.

In [8]:
# ============================================================
# CELL 8 — FBCSP + LDA
# ============================================================

def filter_epochs_band(
    X,
    low_hz,
    high_hz,
    sfreq=160.0,
):
    """
    Deterministic zero-phase filtering for baseline features.

    This is a feature-extraction transform, not a replacement for
    Module 5's primary continuous preprocessing.
    """

    info = mne.create_info(
        ch_names=[
            f"EEG{i:02d}"
            for i in range(22)
        ],
        sfreq=sfreq,
        ch_types=[
            "eeg"
        ] * 22,
    )

    raw = mne.io.RawArray(
        X.reshape(
            22,
            -1,
        ).astype(
            np.float64
        ),
        info,
        verbose="ERROR",
    )

    try:
        raw.filter(
            l_freq=low_hz,
            h_freq=high_hz,
            method="fir",
            phase="zero",
            fir_design="firwin",
            verbose="ERROR",
        )

        filtered = raw.get_data()

    finally:
        raw.close()

    return filtered.reshape(
        X.shape
    ).astype(
        np.float32
    )


def filter_epochs_batch_band(
    X,
    low_hz,
    high_hz,
    sfreq=160.0,
):
    """
    Filter each epoch independently.
    Used only for FBCSP feature construction.
    """

    outputs = []

    for epoch in X:

        outputs.append(
            filter_epochs_band(
                epoch[None, ...],
                low_hz,
                high_hz,
                sfreq,
            )[0]
        )

    return np.stack(
        outputs
    )


def fit_fbcsp_lda(
    X_train,
    y_train,
):

    fitted_bands = []
    feature_blocks_train = []

    for band_id, (
        low_hz,
        high_hz,
    ) in enumerate(FBCSP_BANDS):

        X_band = (
            filter_epochs_batch_band(
                X_train,
                low_hz,
                high_hz,
            )
        )

        csp = CSP(
            n_components=FBCSP_N_COMPONENTS,
            reg="oas",
            log=True,
            norm_trace=False,
            transform_into="average_power",
        )

        features = csp.fit_transform(
            X_band,
            y_train,
        )

        fitted_bands.append(
            {
                "band_id": band_id,
                "low_hz": low_hz,
                "high_hz": high_hz,
                "csp": csp,
            }
        )

        feature_blocks_train.append(
            features
        )

    train_features = np.concatenate(
        feature_blocks_train,
        axis=1,
    )

    lda = LinearDiscriminantAnalysis(
        solver="lsqr",
        shrinkage="auto",
    )

    lda.fit(
        train_features,
        y_train,
    )

    return fitted_bands, lda


def predict_fbcsp_lda(
    fitted_bands,
    lda,
    X_test,
):

    blocks = []

    for item in fitted_bands:

        X_band = (
            filter_epochs_batch_band(
                X_test,
                item["low_hz"],
                item["high_hz"],
            )
        )

        features = item[
            "csp"
        ].transform(
            X_band
        )

        blocks.append(
            features
        )

    test_features = np.concatenate(
        blocks,
        axis=1,
    )

    return lda.predict(
        test_features
    )


print("FBCSP + LDA implementation: READY")

FBCSP + LDA implementation: READY


## Cell 9 — Log-Euclidean covariance + LDA

This baseline avoids an external pyRiemann dependency.

For every epoch:

1. shrinkage covariance using Ledoit-Wolf;
2. matrix logarithm via eigen-decomposition;
3. vectorize the upper triangle.

The transform is learned from the source training data only through the LDA
classifier; covariance construction itself is per-epoch and label-free.

In [9]:
# ============================================================
# CELL 9 — LOG-EUCLIDEAN COVARIANCE + LDA
# ============================================================

def covariance_log_vector(
    epoch,
):
    """
    Log-Euclidean SPD covariance representation.
    """

    X = np.asarray(
        epoch,
        dtype=np.float64,
    )

    # Center each channel.
    X = (
        X
        - X.mean(
            axis=1,
            keepdims=True,
        )
    )

    # Ledoit-Wolf shrinkage covariance.
    lw = LedoitWolf(
        assume_centered=True
    )

    cov = lw.fit(
        X.T
    ).covariance_

    cov = (
        cov
        + cov.T
    ) / 2.0

    eigvals, eigvecs = eigh(
        cov
    )

    eigvals = np.maximum(
        eigvals,
        1e-10,
    )

    log_cov = (
        eigvecs
        @ np.diag(
            np.log(
                eigvals
            )
        )
        @ eigvecs.T
    )

    # Upper triangular vectorization.
    idx = np.triu_indices(
        log_cov.shape[0]
    )

    return log_cov[
        idx
    ]


def riemannian_features(
    X,
):

    return np.stack(
        [
            covariance_log_vector(
                epoch
            )
            for epoch in X
        ]
    )


def fit_riemannian_lda(
    X_train,
    y_train,
):

    train_features = (
        riemannian_features(
            X_train
        )
    )

    lda = LinearDiscriminantAnalysis(
        solver="lsqr",
        shrinkage="auto",
    )

    lda.fit(
        train_features,
        y_train,
    )

    return lda


def predict_riemannian_lda(
    lda,
    X_test,
):

    features = (
        riemannian_features(
            X_test
        )
    )

    return lda.predict(
        features
    )


print(
    "Log-Euclidean covariance + LDA: READY"
)

Log-Euclidean covariance + LDA: READY


## Cell 10 — EEGNet reference architecture

This is a compact EEGNet-style reference, not the proposed domain-generalized
architecture.

Input:
`(B, 1, 22, 640)`

The network is deliberately small enough for MacBook M4 MPS/CPU execution.

In [10]:
# ============================================================
# CELL 10 — EEGNET REFERENCE
# ============================================================

class EEGNetReference(
    nn.Module
):

    def __init__(
        self,
        n_channels=22,
        n_times=640,
        n_classes=3,
        dropout=0.25,
    ):

        super().__init__()

        self.temporal = nn.Sequential(
            nn.Conv2d(
                1,
                8,
                kernel_size=(
                    1,
                    64,
                ),
                padding=(
                    0,
                    32,
                ),
                bias=False,
            ),
            nn.BatchNorm2d(
                8
            ),
        )

        self.depthwise = nn.Sequential(
            nn.Conv2d(
                8,
                16,
                kernel_size=(
                    n_channels,
                    1,
                ),
                groups=8,
                bias=False,
            ),
            nn.BatchNorm2d(
                16
            ),
            nn.ELU(),
            nn.AvgPool2d(
                kernel_size=(
                    1,
                    4,
                )
            ),
            nn.Dropout(
                dropout
            ),
        )

        self.separable = nn.Sequential(
            nn.Conv2d(
                16,
                16,
                kernel_size=(
                    1,
                    16,
                ),
                padding=(
                    0,
                    8,
                ),
                groups=16,
                bias=False,
            ),
            nn.Conv2d(
                16,
                16,
                kernel_size=(
                    1,
                    1,
                ),
                bias=False,
            ),
            nn.BatchNorm2d(
                16
            ),
            nn.ELU(),
            nn.AvgPool2d(
                kernel_size=(
                    1,
                    8,
                )
            ),
            nn.Dropout(
                dropout
            ),
        )

        with torch.no_grad():
            dummy = torch.zeros(
                1,
                1,
                n_channels,
                n_times,
            )

            shape = (
                self.separable(
                    self.depthwise(
                        self.temporal(
                            dummy
                        )
                    )
                )
                .shape
            )

        self.classifier = nn.Linear(
            int(
                np.prod(
                    shape[1:]
                )
            ),
            n_classes,
        )

    def forward(
        self,
        x,
    ):

        # x shape:
        # (B,22,640)
        if x.ndim == 3:
            x = x.unsqueeze(1)

        x = self.temporal(
            x
        )

        x = self.depthwise(
            x
        )

        x = self.separable(
            x
        )

        x = x.flatten(
            start_dim=1
        )

        return self.classifier(
            x
        )


print(
    "EEGNet reference parameters:"
)

test_model = EEGNetReference()

print(
    sum(
        p.numel()
        for p in test_model.parameters()
    )
)

print(
    "EEGNet architecture: PASS"
)

EEGNet reference parameters:
2419
EEGNet architecture: PASS


## Cell 11 — EEGNet training utility

Training uses only source training data.

Validation is created from source subjects only and is used for early stopping.
The target subject is never used for model selection.

In [11]:
# ============================================================
# CELL 11 — EEGNET TRAINING
# ============================================================

def train_eegnet(
    X_train,
    y_train,
    source_subjects,
    seed=SEED,
):

    torch.manual_seed(
        seed
    )

    if DEVICE.type == "mps":
        torch.mps.manual_seed(
            seed
        )

    # --------------------------------------------------------
    # Deterministic source-only validation split by subject.
    # Use the final source subject as validation subject.
    # This keeps validation subject-independent.
    # --------------------------------------------------------

    unique_subjects = sorted(
        set(
            str(s)
            for s in source_subjects
        )
    )

    if len(unique_subjects) < 2:
        raise ValueError(
            "EEGNet requires at least two source subjects "
            "for a subject-level validation split."
        )

    val_subject = unique_subjects[
        -1
    ]

    train_mask = np.array([
        str(s) != val_subject
        for s in source_subjects
    ])

    val_mask = ~train_mask

    X_fit = X_train[
        train_mask
    ]

    y_fit = y_train[
        train_mask
    ]

    X_val = X_train[
        val_mask
    ]

    y_val = y_train[
        val_mask
    ]

    fit_subjects = [
        str(s)
        for i, s in enumerate(
            source_subjects
        )
        if train_mask[i]
    ]

    # --------------------------------------------------------
    # Fit normalization only on training subjects.
    # --------------------------------------------------------

    normalizer = (
        FoldRobustNormalizer()
        .fit(
            X_fit,
            fit_subjects,
        )
    )

    normalizer.assert_target_excluded(
        val_subject
    )

    X_fit = normalizer.transform(
        X_fit
    ).astype(
        np.float32
    )

    X_val = normalizer.transform(
        X_val
    ).astype(
        np.float32
    )

    # --------------------------------------------------------
    # Torch datasets
    # --------------------------------------------------------

    train_ds = TensorDataset(
        torch.from_numpy(
            X_fit
        ),
        torch.from_numpy(
            y_fit.astype(
                np.int64
            )
        ),
    )

    val_ds = TensorDataset(
        torch.from_numpy(
            X_val
        ),
        torch.from_numpy(
            y_val.astype(
                np.int64
            )
        ),
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=EEGNET_BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=False,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=EEGNET_BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=False,
    )

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    model = EEGNetReference(
        n_channels=22,
        n_times=640,
        n_classes=3,
    ).to(
        DEVICE
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=EEGNET_LR,
        weight_decay=EEGNET_WEIGHT_DECAY,
    )

    criterion = nn.CrossEntropyLoss()

    best_state = None
    best_val = np.inf
    patience_count = 0

    history = []

    for epoch in range(
        EEGNET_EPOCHS
    ):

        model.train()

        train_losses = []

        for xb, yb in train_loader:

            xb = xb.to(
                DEVICE
            )

            yb = yb.to(
                DEVICE
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            logits = model(
                xb
            )

            loss = criterion(
                logits,
                yb,
            )

            loss.backward()

            optimizer.step()

            train_losses.append(
                float(
                    loss.detach()
                    .cpu()
                    .item()
                )
            )

        # ----------------------------------------------------
        # Validation
        # ----------------------------------------------------

        model.eval()

        val_losses = []

        with torch.no_grad():

            for xb, yb in val_loader:

                xb = xb.to(
                    DEVICE
                )

                yb = yb.to(
                    DEVICE
                )

                logits = model(
                    xb
                )

                loss = criterion(
                    logits,
                    yb,
                )

                val_losses.append(
                    float(
                        loss.detach()
                        .cpu()
                        .item()
                    )
                )

        mean_train_loss = float(
            np.mean(
                train_losses
            )
        )

        mean_val_loss = float(
            np.mean(
                val_losses
            )
        )

        history.append({
            "epoch": epoch + 1,
            "train_loss": mean_train_loss,
            "val_loss": mean_val_loss,
        })

        if mean_val_loss < best_val:

            best_val = mean_val_loss

            best_state = {
                k: v.detach()
                .cpu()
                .clone()
                for k, v in (
                    model.state_dict()
                    .items()
                )
            }

            patience_count = 0

        else:

            patience_count += 1

        if (
            patience_count
            >= EEGNET_PATIENCE
        ):
            break

    if best_state is not None:
        model.load_state_dict(
            best_state
        )

    return (
        model,
        normalizer,
        pd.DataFrame(history),
    )


def predict_eegnet(
    model,
    normalizer,
    X_test,
):

    X_norm = normalizer.transform(
        X_test
    ).astype(
        np.float32
    )

    ds = TensorDataset(
        torch.from_numpy(
            X_norm
        )
    )

    loader = DataLoader(
        ds,
        batch_size=EEGNET_BATCH_SIZE,
        shuffle=False,
        num_workers=0,
    )

    predictions = []

    model.eval()

    with torch.no_grad():

        for (xb,) in loader:

            xb = xb.to(
                DEVICE
            )

            logits = model(
                xb
            )

            predictions.extend(
                logits.argmax(
                    dim=1
                )
                .cpu()
                .numpy()
                .tolist()
            )

    return np.asarray(
        predictions,
        dtype=np.int64,
    )


print(
    "EEGNet training utilities: READY"
)

EEGNet training utilities: READY


## Cell 12 — Generic classical fold runner

In [12]:
# ============================================================
# CELL 12 — GENERIC CLASSICAL FOLD RUNNER
# ============================================================

def run_classical_fold(
    baseline_name,
    X_train,
    y_train,
    X_test,
    y_test,
):

    start = time.time()

    if baseline_name == "CSP_LDA":

        model_a, model_b = fit_csp_lda(
            X_train,
            y_train,
        )

        y_pred = predict_csp_lda(
            model_a,
            model_b,
            X_test,
        )

    elif baseline_name == "FBCSP_LDA":

        model_a, model_b = fit_fbcsp_lda(
            X_train,
            y_train,
        )

        y_pred = predict_fbcsp_lda(
            model_a,
            model_b,
            X_test,
        )

    elif baseline_name == "RIEMANNIAN_LDA":

        model_a = fit_riemannian_lda(
            X_train,
            y_train,
        )

        y_pred = predict_riemannian_lda(
            model_a,
            X_test,
        )

    else:

        raise ValueError(
            baseline_name
        )

    metrics = compute_metrics(
        y_test,
        y_pred,
    )

    metrics.update({
        "baseline": baseline_name,
        "runtime_sec": float(
            time.time()
            - start
        ),
        "n_train": int(
            len(X_train)
        ),
        "n_test": int(
            len(X_test)
        ),
    })

    return (
        metrics,
        y_pred,
    )


print(
    "Classical fold runner: PASS"
)

Classical fold runner: PASS


## Cell 13 — Convert labels and deterministic result paths

In [13]:
# ============================================================
# CELL 13 — LABELS + RESULT PATHS
# ============================================================

def load_fold_data(
    store,
    train_idx,
    test_idx,
):
    X_train = store.get_X(
        train_idx
    )

    X_test = store.get_X(
        test_idx
    )

    train_meta = cache_meta_df.loc[
        train_idx
    ]

    test_meta = cache_meta_df.loc[
        test_idx
    ]

    y_train = np.asarray([
        CLASS_TO_ID[
            x
        ]
        for x in train_meta[
            "harmonized_class"
        ]
    ], dtype=np.int64)

    y_test = np.asarray([
        CLASS_TO_ID[
            x
        ]
        for x in test_meta[
            "harmonized_class"
        ]
    ], dtype=np.int64)

    train_subjects = (
        train_meta[
            "subject"
        ]
        .astype(str)
        .tolist()
    )

    test_subjects = (
        test_meta[
            "subject"
        ]
        .astype(str)
        .tolist()
    )

    return (
        X_train,
        y_train,
        X_test,
        y_test,
        train_subjects,
        test_subjects,
    )


RESULT_FILES = {
    "CSP_LDA": BASELINE_ROOT / "csp_lda_results.csv",
    "FBCSP_LDA": BASELINE_ROOT / "fbcsp_lda_results.csv",
    "RIEMANNIAN_LDA": BASELINE_ROOT / "riemannian_lda_results.csv",
    "EEGNET": BASELINE_ROOT / "eegnet_results.csv",
}

print(
    "Result paths configured."
)

Result paths configured.


## Cell 14 — Run within-dataset LOSO classical baselines

This cell executes CSP, FBCSP and Riemannian baselines on:

- BCI-IV-2a 9-fold LOSO
- EEGMMIDB 109-fold LOSO

Results are saved after each fold and can be resumed.

In [14]:
# ============================================================
# CELL 14 — WITHIN-DATASET CLASSICAL LOSO
# ============================================================

CLASSICAL_BASELINES = []

if RUN_CSP:
    CLASSICAL_BASELINES.append(
        "CSP_LDA"
    )

if RUN_FBCSP:
    CLASSICAL_BASELINES.append(
        "FBCSP_LDA"
    )

if RUN_RIEMANNIAN:
    CLASSICAL_BASELINES.append(
        "RIEMANNIAN_LDA"
    )


def run_within_classical(
    baseline_name,
):

    result_path = RESULT_FILES[
        baseline_name
    ]

    if result_path.exists():
        existing = pd.read_csv(
            result_path
        )
    else:
        existing = pd.DataFrame()

    done_keys = set()

    if len(existing):
        done_keys = set(
            (
                existing["dataset"].astype(str)
                + "::"
                + existing["target_subject"].astype(str)
            ).tolist()
        )

    rows = []

    folds = within_loso_df.copy()

    if SMOKE_TEST:
        folds = (
            folds
            .groupby(
                "dataset",
                group_keys=False,
            )
            .head(
                SMOKE_FOLDS_PER_PROTOCOL
            )
        )

    with HDF5Store(
        CACHE_PATH
    ) as store:

        for _, fold in folds.iterrows():

            key = (
                str(fold["dataset"])
                + "::"
                + str(fold["target_subject"])
            )

            if key in done_keys:
                continue

            train_idx = safe_protocol_indices(
                fold[
                    "train_indices_json"
                ]
            )

            test_idx = safe_protocol_indices(
                fold[
                    "test_indices_json"
                ]
            )

            (
                X_train,
                y_train,
                X_test,
                y_test,
                train_subjects,
                test_subjects,
            ) = load_fold_data(
                store,
                train_idx,
                test_idx,
            )

            assert (
                fold["target_subject"]
                not in set(
                    train_subjects
                )
            )

            metrics, y_pred = (
                run_classical_fold(
                    baseline_name,
                    X_train,
                    y_train,
                    X_test,
                    y_test,
                )
            )

            row = {
                **metrics,
                "protocol": "within_dataset_loso",
                "dataset": fold[
                    "dataset"
                ],
                "target_subject": fold[
                    "target_subject"
                ],
                "fold_id": int(
                    fold["fold_id"]
                ),
                "confusion_matrix": json.dumps(
                    confusion_matrix(
                        y_test,
                        y_pred,
                        labels=[0, 1, 2],
                    ).tolist()
                ),
            }

            rows.append(row)

            existing = pd.concat(
                [
                    existing,
                    pd.DataFrame([row]),
                ],
                ignore_index=True,
            )

            existing.to_csv(
                result_path,
                index=False,
            )

            print(
                f"{baseline_name} | "
                f"{fold['dataset']} | "
                f"{fold['target_subject']} | "
                f"Acc={row['accuracy']:.4f}"
            )

    return pd.read_csv(
        result_path
    )


within_classical_results = {}

for baseline_name in CLASSICAL_BASELINES:

    print("\n" + "=" * 78)
    print(
        "RUNNING:",
        baseline_name,
    )
    print("=" * 78)

    within_classical_results[
        baseline_name
    ] = run_within_classical(
        baseline_name
    )


RUNNING: CSP_LDA
Computing rank from data with rank=None
    Using tolerance 5.2e-05 (2.2e-16 eps * 22 dim * 1.1e+10  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=0 covariance using OAS
Done.
Estimating class=1 covariance using OAS
Done.
Estimating class=2 covariance using OAS
Done.
CSP_LDA | BCI-IV-2a | S01 | Acc=0.6389
Computing rank from data with rank=None
    Using tolerance 5.3e-05 (2.2e-16 eps * 22 dim * 1.1e+10  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=0 covariance using OAS
Done.
Estimating class=1 covariance using OAS
Done.
Estimating class=2 covariance using OAS
Done.
CSP_LDA | BCI-IV-2a | S02 | Acc=0.3426
Computing rank from data with rank=None
    Using tolerance 5.2e-05 (2.2e-16 eps * 22 dim * 1.1e+10  max singular value)
    E

## Cell 15 — Summarize within-dataset classical baselines

In [16]:
# ============================================================
# CELL 15 — WITHIN-DATASET CLASSICAL SUMMARY
# ============================================================

within_summary_rows = []

for baseline_name, df in (
    within_classical_results.items()
):

    for dataset_name, sub in (
        df.groupby("dataset")
    ):

        within_summary_rows.append({
            "baseline": baseline_name,
            "dataset": dataset_name,
            "folds": len(sub),
            "accuracy_mean": sub[
                "accuracy"
            ].mean(),
            "accuracy_std": sub[
                "accuracy"
            ].std(ddof=1),
            "balanced_accuracy_mean": sub[
                "balanced_accuracy"
            ].mean(),
            "balanced_accuracy_std": sub[
                "balanced_accuracy"
            ].std(ddof=1),
            "macro_f1_mean": sub[
                "macro_f1"
            ].mean(),
            "macro_f1_std": sub[
                "macro_f1"
            ].std(ddof=1),
        })

within_summary_df = pd.DataFrame(
    within_summary_rows
)

display(
    within_summary_df
)

within_summary_df.to_csv(
    BASELINE_ROOT
    / "within_dataset_classical_summary.csv",
    index=False,
)

,baseline,dataset,folds,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,macro_f1_mean,macro_f1_std
0,CSP_LDA,BCI-IV-2a,9,0.433642,0.104141,0.433642,0.104141,0.365735,0.123491
1,CSP_LDA,EEGMMIDB,109,0.387589,0.085967,0.389318,0.085154,0.298073,0.120181
2,FBCSP_LDA,BCI-IV-2a,9,0.424383,0.080154,0.424383,0.080154,0.353535,0.105529
3,FBCSP_LDA,EEGMMIDB,109,0.432495,0.117024,0.433083,0.116829,0.376585,0.144344
4,RIEMANNIAN_LDA,BCI-IV-2a,9,0.383230,0.060442,0.383230,0.060442,0.272271,0.121731
5,RIEMANNIAN_LDA,EEGMMIDB,109,0.406456,0.103342,0.405257,0.104071,0.335243,0.133881


## Cell 16 — Cross-dataset classical baselines

This is the primary zero-calibration domain-generalization baseline.

For each target subject:
- source dataset is fully available for fitting;
- target subject is used only for final evaluation.

No target-derived normalization, CSP, or classifier parameters are permitted.

In [17]:
# ============================================================
# CELL 16 — CROSS-DATASET CLASSICAL BASELINES
# ============================================================

def run_transfer_classical(
    baseline_name,
):

    result_path = (
        BASELINE_ROOT
        / f"{baseline_name.lower()}_transfer_results.csv"
    )

    if result_path.exists():
        existing = pd.read_csv(
            result_path
        )
    else:
        existing = pd.DataFrame()

    done_keys = set()

    if len(existing):
        done_keys = set(
            (
                existing["source_dataset"].astype(str)
                + "->"
                + existing["target_dataset"].astype(str)
                + "::"
                + existing["target_subject"].astype(str)
            ).tolist()
        )

    rows = []

    folds = transfer_df.copy()

    if SMOKE_TEST:
        folds = (
            folds
            .groupby(
                [
                    "source_dataset",
                    "target_dataset",
                ],
                group_keys=False,
            )
            .head(
                SMOKE_FOLDS_PER_PROTOCOL
            )
        )

    with HDF5Store(
        CACHE_PATH
    ) as store:

        for _, fold in folds.iterrows():

            key = (
                str(fold["source_dataset"])
                + "->"
                + str(fold["target_dataset"])
                + "::"
                + str(fold["target_subject"])
            )

            if key in done_keys:
                continue

            train_idx = safe_protocol_indices(
                fold[
                    "train_indices_json"
                ]
            )

            test_idx = safe_protocol_indices(
                fold[
                    "test_indices_json"
                ]
            )

            (
                X_train,
                y_train,
                X_test,
                y_test,
                train_subjects,
                test_subjects,
            ) = load_fold_data(
                store,
                train_idx,
                test_idx,
            )

            assert set(
                cache_meta_df.loc[
                    train_idx,
                    "dataset",
                ]
            ) == {
                fold["source_dataset"]
            }

            assert set(
                cache_meta_df.loc[
                    test_idx,
                    "dataset",
                ]
            ) == {
                fold["target_dataset"]
            }

            assert (
                fold["target_subject"]
                not in set(
                    train_subjects
                )
            )

            metrics, y_pred = (
                run_classical_fold(
                    baseline_name,
                    X_train,
                    y_train,
                    X_test,
                    y_test,
                )
            )

            row = {
                **metrics,
                "protocol": "cross_dataset_zero_calibration",
                "source_dataset": fold[
                    "source_dataset"
                ],
                "target_dataset": fold[
                    "target_dataset"
                ],
                "target_subject": fold[
                    "target_subject"
                ],
                "fold_id": int(
                    fold["fold_id"]
                ),
                "confusion_matrix": json.dumps(
                    confusion_matrix(
                        y_test,
                        y_pred,
                        labels=[0, 1, 2],
                    ).tolist()
                ),
            }

            rows.append(row)

            existing = pd.concat(
                [
                    existing,
                    pd.DataFrame([row]),
                ],
                ignore_index=True,
            )

            existing.to_csv(
                result_path,
                index=False,
            )

            print(
                f"{baseline_name} | "
                f"{fold['source_dataset']} -> "
                f"{fold['target_dataset']} | "
                f"{fold['target_subject']} | "
                f"Acc={row['accuracy']:.4f}"
            )

    return pd.read_csv(
        result_path
    )


transfer_classical_results = {}

for baseline_name in CLASSICAL_BASELINES:

    print("\n" + "=" * 78)
    print(
        "TRANSFER:",
        baseline_name,
    )
    print("=" * 78)

    transfer_classical_results[
        baseline_name
    ] = run_transfer_classical(
        baseline_name
    )


TRANSFER: CSP_LDA
Computing rank from data with rank=None
    Using tolerance 5.4e-05 (2.2e-16 eps * 22 dim * 1.1e+10  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=0 covariance using OAS
Done.
Estimating class=1 covariance using OAS
Done.
Estimating class=2 covariance using OAS
Done.
CSP_LDA | BCI-IV-2a -> EEGMMIDB | S001 | Acc=0.4638
Computing rank from data with rank=None
    Using tolerance 5.4e-05 (2.2e-16 eps * 22 dim * 1.1e+10  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=0 covariance using OAS
Done.
Estimating class=1 covariance using OAS
Done.
Estimating class=2 covariance using OAS
Done.
CSP_LDA | BCI-IV-2a -> EEGMMIDB | S002 | Acc=0.5152
Computing rank from data with rank=None
    Using tolerance 5.4e-05 (2.2e-16 eps * 22 dim * 1.1e+10

## Cell 17 — Summarize cross-dataset classical results

In [18]:
# ============================================================
# CELL 17 — CROSS-DATASET CLASSICAL SUMMARY
# ============================================================

transfer_summary_rows = []

for baseline_name, df in (
    transfer_classical_results.items()
):

    for (
        source_dataset,
        target_dataset,
    ), sub in df.groupby(
        [
            "source_dataset",
            "target_dataset",
        ]
    ):

        transfer_summary_rows.append({
            "baseline": baseline_name,
            "source_dataset": source_dataset,
            "target_dataset": target_dataset,
            "folds": len(sub),
            "accuracy_mean": sub[
                "accuracy"
            ].mean(),
            "accuracy_std": sub[
                "accuracy"
            ].std(ddof=1),
            "balanced_accuracy_mean": sub[
                "balanced_accuracy"
            ].mean(),
            "balanced_accuracy_std": sub[
                "balanced_accuracy"
            ].std(ddof=1),
            "macro_f1_mean": sub[
                "macro_f1"
            ].mean(),
            "macro_f1_std": sub[
                "macro_f1"
            ].std(ddof=1),
        })

transfer_summary_df = pd.DataFrame(
    transfer_summary_rows
)

display(
    transfer_summary_df
)

transfer_summary_df.to_csv(
    BASELINE_ROOT
    / "cross_dataset_classical_summary.csv",
    index=False,
)

,baseline,source_dataset,target_dataset,folds,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,macro_f1_mean,macro_f1_std
0,CSP_LDA,BCI-IV-2a,EEGMMIDB,109,0.383361,0.090951,0.383617,0.089795,0.277252,0.117027
1,CSP_LDA,EEGMMIDB,BCI-IV-2a,9,0.352881,0.028891,0.352881,0.028891,0.236047,0.062400
2,FBCSP_LDA,BCI-IV-2a,EEGMMIDB,109,0.371769,0.087208,0.371531,0.086879,0.271719,0.116229
3,FBCSP_LDA,EEGMMIDB,BCI-IV-2a,9,0.469136,0.104757,0.469136,0.104757,0.432522,0.125298
4,RIEMANNIAN_LDA,BCI-IV-2a,EEGMMIDB,109,0.365795,0.066568,0.364532,0.067222,0.273401,0.097557
5,RIEMANNIAN_LDA,EEGMMIDB,BCI-IV-2a,9,0.401749,0.097629,0.401749,0.097629,0.292008,0.162829


## Cell 18 — Aggregate confusion matrices for classical baselines

In [19]:
# ============================================================
# CELL 18 — AGGREGATE CONFUSION MATRICES
# ============================================================

def aggregate_confusion_matrix(
    df,
):

    cm = np.zeros(
        (
            3,
            3,
        ),
        dtype=np.int64,
    )

    for value in df[
        "confusion_matrix"
    ]:

        arr = np.asarray(
            json.loads(value),
            dtype=np.int64,
        )

        cm += arr

    return cm


all_confusion_rows = []

for baseline_name, df in (
    within_classical_results.items()
):

    for dataset_name, sub in (
        df.groupby("dataset")
    ):

        cm = aggregate_confusion_matrix(
            sub
        )

        all_confusion_rows.append({
            "baseline": baseline_name,
            "protocol": "within_dataset_loso",
            "source_dataset": dataset_name,
            "target_dataset": dataset_name,
            "confusion_matrix": cm.tolist(),
        })


for baseline_name, df in (
    transfer_classical_results.items()
):

    for (
        source_dataset,
        target_dataset,
    ), sub in df.groupby(
        [
            "source_dataset",
            "target_dataset",
        ]
    ):

        cm = aggregate_confusion_matrix(
            sub
        )

        all_confusion_rows.append({
            "baseline": baseline_name,
            "protocol": "cross_dataset_zero_calibration",
            "source_dataset": source_dataset,
            "target_dataset": target_dataset,
            "confusion_matrix": cm.tolist(),
        })


aggregate_cm_df = pd.DataFrame(
    all_confusion_rows
)

display(
    aggregate_cm_df
)

aggregate_cm_df.to_csv(
    BASELINE_ROOT
    / "aggregate_confusion_matrices.csv",
    index=False,
)

,baseline,protocol,source_dataset,target_dataset,confusion_matrix
0,CSP_LDA,within_dataset_loso,BCI-IV-2a,BCI-IV-2a,"[[260, 186, 202], [136, 344, 168], [203, 206, ..."
1,CSP_LDA,within_dataset_loso,EEGMMIDB,EEGMMIDB,"[[635, 957, 887], [546, 1140, 752], [515, 859,..."
2,FBCSP_LDA,within_dataset_loso,BCI-IV-2a,BCI-IV-2a,"[[263, 159, 226], [124, 337, 187], [235, 188, ..."
3,FBCSP_LDA,within_dataset_loso,EEGMMIDB,EEGMMIDB,"[[1033, 793, 653], [653, 1197, 588], [686, 811..."
4,RIEMANNIAN_LDA,within_dataset_loso,BCI-IV-2a,BCI-IV-2a,"[[503, 27, 118], [470, 114, 64], [462, 58, 128]]"
5,RIEMANNIAN_LDA,within_dataset_loso,EEGMMIDB,EEGMMIDB,"[[1393, 550, 536], [1037, 943, 458], [1191, 60..."
6,CSP_LDA,cross_dataset_zero_calibration,BCI-IV-2a,EEGMMIDB,"[[1156, 1254, 69], [837, 1537, 64], [934, 1389..."
7,CSP_LDA,cross_dataset_zero_calibration,EEGMMIDB,BCI-IV-2a,"[[14, 84, 550], [19, 109, 520], [14, 71, 563]]"
8,FBCSP_LDA,cross_dataset_zero_calibration,BCI-IV-2a,EEGMMIDB,"[[639, 376, 1464], [420, 576, 1442], [499, 432..."
9,FBCSP_LDA,cross_dataset_zero_calibration,EEGMMIDB,BCI-IV-2a,"[[264, 120, 264], [118, 252, 278], [136, 116, ..."


## Cell 19 — EEGNet within-dataset LOSO

This executes the compact neural reference.

For each target subject:
- source training subjects are used for normalization fitting and model fitting;
- one source subject is held out for early-stopping validation;
- the target subject remains completely unseen until final evaluation.

Results are appended to CSV after every fold.

In [20]:
# ============================================================
# CELL 19 — EEGNET WITHIN-DATASET LOSO
# ============================================================

def run_eegnet_within():

    result_path = (
        BASELINE_ROOT
        / "eegnet_within_loso_results.csv"
    )

    history_root = (
        BASELINE_ROOT
        / "eegnet_histories"
    )

    history_root.mkdir(
        exist_ok=True
    )

    if result_path.exists():
        existing = pd.read_csv(
            result_path
        )
    else:
        existing = pd.DataFrame()

    done_keys = set()

    if len(existing):
        done_keys = set(
            (
                existing["dataset"].astype(str)
                + "::"
                + existing["target_subject"].astype(str)
            ).tolist()
        )

    folds = within_loso_df.copy()

    if SMOKE_TEST:
        folds = (
            folds
            .groupby(
                "dataset",
                group_keys=False,
            )
            .head(
                SMOKE_FOLDS_PER_PROTOCOL
            )
        )

    with HDF5Store(
        CACHE_PATH
    ) as store:

        for _, fold in folds.iterrows():

            key = (
                str(fold["dataset"])
                + "::"
                + str(fold["target_subject"])
            )

            if key in done_keys:
                continue

            train_idx = safe_protocol_indices(
                fold[
                    "train_indices_json"
                ]
            )

            test_idx = safe_protocol_indices(
                fold[
                    "test_indices_json"
                ]
            )

            X_train = store.get_X(
                train_idx
            )

            X_test = store.get_X(
                test_idx
            )

            train_meta = cache_meta_df.loc[
                train_idx
            ]

            test_meta = cache_meta_df.loc[
                test_idx
            ]

            y_train = np.asarray([
                CLASS_TO_ID[
                    x
                ]
                for x in train_meta[
                    "harmonized_class"
                ]
            ], dtype=np.int64)

            y_test = np.asarray([
                CLASS_TO_ID[
                    x
                ]
                for x in test_meta[
                    "harmonized_class"
                ]
            ], dtype=np.int64)

            source_subjects = (
                train_meta[
                    "subject"
                ]
                .astype(str)
                .tolist()
            )

            assert (
                fold["target_subject"]
                not in set(
                    source_subjects
                )
            )

            start = time.time()

            model, normalizer, history = (
                train_eegnet(
                    X_train,
                    y_train,
                    source_subjects,
                    seed=SEED + int(
                        fold["fold_id"]
                    ),
                )
            )

            normalizer.assert_target_excluded(
                fold["target_subject"]
            )

            y_pred = predict_eegnet(
                model,
                normalizer,
                X_test,
            )

            metrics = compute_metrics(
                y_test,
                y_pred,
            )

            row = {
                **metrics,
                "protocol": "within_dataset_loso",
                "dataset": fold[
                    "dataset"
                ],
                "target_subject": fold[
                    "target_subject"
                ],
                "fold_id": int(
                    fold["fold_id"]
                ),
                "runtime_sec": float(
                    time.time()
                    - start
                ),
                "best_val_loss": float(
                    history["val_loss"].min()
                ),
                "epochs_run": int(
                    len(history)
                ),
                "confusion_matrix": json.dumps(
                    confusion_matrix(
                        y_test,
                        y_pred,
                        labels=[0, 1, 2],
                    ).tolist()
                ),
            }

            existing = pd.concat(
                [
                    existing,
                    pd.DataFrame([row]),
                ],
                ignore_index=True,
            )

            existing.to_csv(
                result_path,
                index=False,
            )

            history.to_csv(
                history_root
                / (
                    f"within_"
                    f"{fold['dataset']}_"
                    f"{fold['target_subject']}.csv"
                ),
                index=False,
            )

            print(
                f"EEGNet | "
                f"{fold['dataset']} | "
                f"{fold['target_subject']} | "
                f"Acc={row['accuracy']:.4f}"
            )

    return pd.read_csv(
        result_path
    )


eegnet_within_results = None

if RUN_EEGNET:
    if EEGNET_FULL_WITHIN_LOSO or SMOKE_TEST:
        eegnet_within_results = (
            run_eegnet_within()
        )

print(
    "EEGNet within-dataset run complete."
    if eegnet_within_results is not None
    else
    "EEGNet within-dataset run skipped."
)

EEGNet | BCI-IV-2a | S01 | Acc=0.6528
EEGNet | BCI-IV-2a | S02 | Acc=0.3657
EEGNet | BCI-IV-2a | S03 | Acc=0.6898
EEGNet | BCI-IV-2a | S04 | Acc=0.4769
EEGNet | BCI-IV-2a | S05 | Acc=0.3657
EEGNet | BCI-IV-2a | S06 | Acc=0.4120
EEGNet | BCI-IV-2a | S07 | Acc=0.3472
EEGNet | BCI-IV-2a | S08 | Acc=0.3750
EEGNet | BCI-IV-2a | S09 | Acc=0.6343
EEGNet | EEGMMIDB | S001 | Acc=0.5072
EEGNet | EEGMMIDB | S002 | Acc=0.3788
EEGNet | EEGMMIDB | S003 | Acc=0.3478
EEGNet | EEGMMIDB | S004 | Acc=0.3824
EEGNet | EEGMMIDB | S005 | Acc=0.3731
EEGNet | EEGMMIDB | S006 | Acc=0.3088
EEGNet | EEGMMIDB | S007 | Acc=0.4559
EEGNet | EEGMMIDB | S008 | Acc=0.3731
EEGNet | EEGMMIDB | S009 | Acc=0.2687
EEGNet | EEGMMIDB | S010 | Acc=0.4179
EEGNet | EEGMMIDB | S011 | Acc=0.3529
EEGNet | EEGMMIDB | S012 | Acc=0.3088
EEGNet | EEGMMIDB | S013 | Acc=0.5000
EEGNet | EEGMMIDB | S014 | Acc=0.3333
EEGNet | EEGMMIDB | S015 | Acc=0.4706
EEGNet | EEGMMIDB | S016 | Acc=0.4091
EEGNet | EEGMMIDB | S017 | Acc=0.3182
EEGNet | EEG

## Cell 20 — EEGNet cross-dataset zero-calibration

Cross-dataset EEGNet is trained on one complete source dataset.

For each target subject:
- source normalization is fitted only on source data;
- source model is trained only on source data;
- target is used only for final prediction.

In [21]:
# ============================================================
# CELL 20 — EEGNET CROSS-DATASET
# ============================================================

def run_eegnet_transfer():

    result_path = (
        BASELINE_ROOT
        / "eegnet_transfer_results.csv"
    )

    history_root = (
        BASELINE_ROOT
        / "eegnet_transfer_histories"
    )

    history_root.mkdir(
        exist_ok=True
    )

    if result_path.exists():
        existing = pd.read_csv(
            result_path
        )
    else:
        existing = pd.DataFrame()

    done_keys = set()

    if len(existing):
        done_keys = set(
            (
                existing["source_dataset"].astype(str)
                + "->"
                + existing["target_dataset"].astype(str)
                + "::"
                + existing["target_subject"].astype(str)
            ).tolist()
        )

    folds = transfer_df.copy()

    if SMOKE_TEST:
        folds = (
            folds
            .groupby(
                [
                    "source_dataset",
                    "target_dataset",
                ],
                group_keys=False,
            )
            .head(
                SMOKE_FOLDS_PER_PROTOCOL
            )
        )

    with HDF5Store(
        CACHE_PATH
    ) as store:

        # ----------------------------------------------------
        # Cache source models in one per-direction training pass
        # only when all target subjects share the same source.
        # ----------------------------------------------------

        source_models = {}

        for direction in [
            (
                "BCI-IV-2a",
                "EEGMMIDB",
            ),
            (
                "EEGMMIDB",
                "BCI-IV-2a",
            ),
        ]:

            direction_source, direction_target = (
                direction
            )

            direction_folds = folds[
                (
                    folds["source_dataset"]
                    == direction_source
                )
                &
                (
                    folds["target_dataset"]
                    == direction_target
                )
            ]

            if len(
                direction_folds
            ) == 0:
                continue

            source_indices = safe_protocol_indices(
                direction_folds.iloc[0][
                    "train_indices_json"
                ]
            )

            X_source = store.get_X(
                source_indices
            )

            source_meta = cache_meta_df.loc[
                source_indices
            ]

            y_source = np.asarray([
                CLASS_TO_ID[
                    x
                ]
                for x in source_meta[
                    "harmonized_class"
                ]
            ], dtype=np.int64)

            source_subjects = (
                source_meta[
                    "subject"
                ]
                .astype(str)
                .tolist()
            )

            print(
                "\nTraining source-direction EEGNet:",
                direction_source,
                "->",
                direction_target,
            )

            start = time.time()

            model, normalizer, history = (
                train_eegnet(
                    X_source,
                    y_source,
                    source_subjects,
                    seed=SEED + 1000,
                )
            )

            source_models[
                direction
            ] = (
                model,
                normalizer,
                history,
            )

            print(
                "Source model trained in",
                round(
                    time.time() - start,
                    2,
                ),
                "sec",
            )


        # ----------------------------------------------------
        # Evaluate every target subject
        # ----------------------------------------------------

        for _, fold in folds.iterrows():

            key = (
                str(fold["source_dataset"])
                + "->"
                + str(fold["target_dataset"])
                + "::"
                + str(fold["target_subject"])
            )

            if key in done_keys:
                continue

            direction = (
                fold["source_dataset"],
                fold["target_dataset"],
            )

            model, normalizer, history = (
                source_models[
                    direction
                ]
            )

            test_idx = safe_protocol_indices(
                fold[
                    "test_indices_json"
                ]
            )

            X_test = store.get_X(
                test_idx
            )

            test_meta = cache_meta_df.loc[
                test_idx
            ]

            y_test = np.asarray([
                CLASS_TO_ID[
                    x
                ]
                for x in test_meta[
                    "harmonized_class"
                ]
            ], dtype=np.int64)

            normalizer.assert_target_excluded(
                fold["target_subject"]
            )

            start = time.time()

            y_pred = predict_eegnet(
                model,
                normalizer,
                X_test,
            )

            metrics = compute_metrics(
                y_test,
                y_pred,
            )

            row = {
                **metrics,
                "protocol": "cross_dataset_zero_calibration",
                "source_dataset": fold[
                    "source_dataset"
                ],
                "target_dataset": fold[
                    "target_dataset"
                ],
                "target_subject": fold[
                    "target_subject"
                ],
                "fold_id": int(
                    fold["fold_id"]
                ),
                "runtime_sec": float(
                    time.time()
                    - start
                ),
                "source_training_epochs": len(
                    source_indices
                ),
                "source_validation_best_loss": float(
                    history["val_loss"].min()
                ),
                "epochs_run": int(
                    len(history)
                ),
                "confusion_matrix": json.dumps(
                    confusion_matrix(
                        y_test,
                        y_pred,
                        labels=[0, 1, 2],
                    ).tolist()
                ),
            }

            existing = pd.concat(
                [
                    existing,
                    pd.DataFrame([row]),
                ],
                ignore_index=True,
            )

            existing.to_csv(
                result_path,
                index=False,
            )

            print(
                f"EEGNet | "
                f"{fold['source_dataset']} -> "
                f"{fold['target_dataset']} | "
                f"{fold['target_subject']} | "
                f"Acc={row['accuracy']:.4f}"
            )

    return pd.read_csv(
        result_path
    )


eegnet_transfer_results = None

if RUN_EEGNET:
    if EEGNET_FULL_CROSS_DATASET or SMOKE_TEST:
        eegnet_transfer_results = (
            run_eegnet_transfer()
        )

print(
    "EEGNet transfer run complete."
    if eegnet_transfer_results is not None
    else
    "EEGNet transfer run skipped."
)


Training source-direction EEGNet: BCI-IV-2a -> EEGMMIDB
Source model trained in 37.3 sec

Training source-direction EEGNet: EEGMMIDB -> BCI-IV-2a
Source model trained in 37.88 sec
EEGNet | BCI-IV-2a -> EEGMMIDB | S001 | Acc=0.3478
EEGNet | BCI-IV-2a -> EEGMMIDB | S002 | Acc=0.5758
EEGNet | BCI-IV-2a -> EEGMMIDB | S003 | Acc=0.3043
EEGNet | BCI-IV-2a -> EEGMMIDB | S004 | Acc=0.3382
EEGNet | BCI-IV-2a -> EEGMMIDB | S005 | Acc=0.3284
EEGNet | BCI-IV-2a -> EEGMMIDB | S006 | Acc=0.3382
EEGNet | BCI-IV-2a -> EEGMMIDB | S007 | Acc=0.4412
EEGNet | BCI-IV-2a -> EEGMMIDB | S008 | Acc=0.4478
EEGNet | BCI-IV-2a -> EEGMMIDB | S009 | Acc=0.3284
EEGNet | BCI-IV-2a -> EEGMMIDB | S010 | Acc=0.3433
EEGNet | BCI-IV-2a -> EEGMMIDB | S011 | Acc=0.3382
EEGNet | BCI-IV-2a -> EEGMMIDB | S012 | Acc=0.3824
EEGNet | BCI-IV-2a -> EEGMMIDB | S013 | Acc=0.3382
EEGNet | BCI-IV-2a -> EEGMMIDB | S014 | Acc=0.3182
EEGNet | BCI-IV-2a -> EEGMMIDB | S015 | Acc=0.4118
EEGNet | BCI-IV-2a -> EEGMMIDB | S016 | Acc=0.3333
EEG

## Cell 21 — Summarize EEGNet results

In [22]:
# ============================================================
# CELL 21 — EEGNET SUMMARY
# ============================================================

eegnet_summary_rows = []

if eegnet_within_results is not None:

    for dataset_name, sub in (
        eegnet_within_results
        .groupby("dataset")
    ):

        eegnet_summary_rows.append({
            "baseline": "EEGNet",
            "protocol": "within_dataset_loso",
            "source_dataset": dataset_name,
            "target_dataset": dataset_name,
            "folds": len(sub),
            "accuracy_mean": sub["accuracy"].mean(),
            "accuracy_std": sub["accuracy"].std(ddof=1),
            "balanced_accuracy_mean": sub[
                "balanced_accuracy"
            ].mean(),
            "balanced_accuracy_std": sub[
                "balanced_accuracy"
            ].std(ddof=1),
            "macro_f1_mean": sub[
                "macro_f1"
            ].mean(),
            "macro_f1_std": sub[
                "macro_f1"
            ].std(ddof=1),
        })


if eegnet_transfer_results is not None:

    for (
        source_dataset,
        target_dataset,
    ), sub in (
        eegnet_transfer_results
        .groupby(
            [
                "source_dataset",
                "target_dataset",
            ]
        )
    ):

        eegnet_summary_rows.append({
            "baseline": "EEGNet",
            "protocol": "cross_dataset_zero_calibration",
            "source_dataset": source_dataset,
            "target_dataset": target_dataset,
            "folds": len(sub),
            "accuracy_mean": sub["accuracy"].mean(),
            "accuracy_std": sub["accuracy"].std(ddof=1),
            "balanced_accuracy_mean": sub[
                "balanced_accuracy"
            ].mean(),
            "balanced_accuracy_std": sub[
                "balanced_accuracy"
            ].std(ddof=1),
            "macro_f1_mean": sub[
                "macro_f1"
            ].mean(),
            "macro_f1_std": sub[
                "macro_f1"
            ].std(ddof=1),
        })


eegnet_summary_df = pd.DataFrame(
    eegnet_summary_rows
)

if len(eegnet_summary_df):
    display(
        eegnet_summary_df
    )

    eegnet_summary_df.to_csv(
        BASELINE_ROOT
        / "eegnet_summary.csv",
        index=False,
    )
else:
    print(
        "No EEGNet results available."
    )

,baseline,protocol,source_dataset,target_dataset,folds,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,macro_f1_mean,macro_f1_std
0,EEGNet,within_dataset_loso,BCI-IV-2a,BCI-IV-2a,9,0.479938,0.140118,0.479938,0.140118,0.419485,0.173127
1,EEGNet,within_dataset_loso,EEGMMIDB,EEGMMIDB,109,0.369738,0.061860,0.368760,0.062206,0.317173,0.083684
2,EEGNet,cross_dataset_zero_calibration,BCI-IV-2a,EEGMMIDB,109,0.387051,0.105275,0.386292,0.105417,0.274838,0.144824
3,EEGNet,cross_dataset_zero_calibration,EEGMMIDB,BCI-IV-2a,9,0.348765,0.023720,0.348765,0.023720,0.215515,0.068584


## Cell 22 — Unified Module 7 baseline table

In [23]:
# ============================================================
# CELL 22 — UNIFIED BASELINE TABLE
# ============================================================

unified_rows = []

if len(within_summary_df):

    for _, row in within_summary_df.iterrows():

        unified_rows.append({
            **row.to_dict(),
            "protocol": "within_dataset_loso",
            "source_dataset": row["dataset"],
            "target_dataset": row["dataset"],
        })


if len(transfer_summary_df):

    for _, row in transfer_summary_df.iterrows():

        unified_rows.append({
            **row.to_dict(),
            "protocol": "cross_dataset_zero_calibration",
        })


if len(eegnet_summary_df):

    unified_rows.extend(
        eegnet_summary_df.to_dict(
            orient="records"
        )
    )


unified_baseline_df = pd.DataFrame(
    unified_rows
)

if len(unified_baseline_df):

    unified_baseline_df = unified_baseline_df[
        [
            "baseline",
            "protocol",
            "source_dataset",
            "target_dataset",
            "folds",
            "accuracy_mean",
            "accuracy_std",
            "balanced_accuracy_mean",
            "balanced_accuracy_std",
            "macro_f1_mean",
            "macro_f1_std",
        ]
    ]

    display(
        unified_baseline_df
    )

    unified_baseline_df.to_csv(
        BASELINE_ROOT
        / "module_7_unified_baseline_summary.csv",
        index=False,
    )
else:

    print(
        "No baseline results available."
    )

,baseline,protocol,source_dataset,target_dataset,folds,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,macro_f1_mean,macro_f1_std
0,CSP_LDA,within_dataset_loso,BCI-IV-2a,BCI-IV-2a,9,0.433642,0.104141,0.433642,0.104141,0.365735,0.123491
1,CSP_LDA,within_dataset_loso,EEGMMIDB,EEGMMIDB,109,0.387589,0.085967,0.389318,0.085154,0.298073,0.120181
2,FBCSP_LDA,within_dataset_loso,BCI-IV-2a,BCI-IV-2a,9,0.424383,0.080154,0.424383,0.080154,0.353535,0.105529
3,FBCSP_LDA,within_dataset_loso,EEGMMIDB,EEGMMIDB,109,0.432495,0.117024,0.433083,0.116829,0.376585,0.144344
4,RIEMANNIAN_LDA,within_dataset_loso,BCI-IV-2a,BCI-IV-2a,9,0.383230,0.060442,0.383230,0.060442,0.272271,0.121731
5,RIEMANNIAN_LDA,within_dataset_loso,EEGMMIDB,EEGMMIDB,109,0.406456,0.103342,0.405257,0.104071,0.335243,0.133881
6,CSP_LDA,cross_dataset_zero_calibration,BCI-IV-2a,EEGMMIDB,109,0.383361,0.090951,0.383617,0.089795,0.277252,0.117027
7,CSP_LDA,cross_dataset_zero_calibration,EEGMMIDB,BCI-IV-2a,9,0.352881,0.028891,0.352881,0.028891,0.236047,0.062400
8,FBCSP_LDA,cross_dataset_zero_calibration,BCI-IV-2a,EEGMMIDB,109,0.371769,0.087208,0.371531,0.086879,0.271719,0.116229
9,FBCSP_LDA,cross_dataset_zero_calibration,EEGMMIDB,BCI-IV-2a,9,0.469136,0.104757,0.469136,0.104757,0.432522,0.125298


## Cell 23 — Baseline confidence and chance-level diagnostics

This module does not perform inferential statistics.

It reports:
- 33.33% three-class chance level;
- fold-level accuracy distributions;
- whether a baseline is above chance descriptively.

Formal confidence intervals and paired statistical tests belong to Module 10.

In [24]:
# ============================================================
# CELL 23 — CHANCE-LEVEL DIAGNOSTICS
# ============================================================

CHANCE_LEVEL = 1.0 / 3.0

print(
    "Three-class chance level:",
    f"{CHANCE_LEVEL * 100:.2f}%"
)

if len(unified_baseline_df):

    chance_table = unified_baseline_df.copy()

    chance_table[
        "mean_accuracy_minus_chance"
    ] = (
        chance_table["accuracy_mean"]
        - CHANCE_LEVEL
    )

    display(
        chance_table[
            [
                "baseline",
                "protocol",
                "source_dataset",
                "target_dataset",
                "accuracy_mean",
                "mean_accuracy_minus_chance",
            ]
        ]
    )

    chance_table.to_csv(
        BASELINE_ROOT
        / "module_7_chance_level_diagnostics.csv",
        index=False,
    )

print(
    "\nChance-level diagnostic: COMPLETE"
)

Three-class chance level: 33.33%


,baseline,protocol,source_dataset,target_dataset,accuracy_mean,mean_accuracy_minus_chance
0,CSP_LDA,within_dataset_loso,BCI-IV-2a,BCI-IV-2a,0.433642,0.100309
1,CSP_LDA,within_dataset_loso,EEGMMIDB,EEGMMIDB,0.387589,0.054256
2,FBCSP_LDA,within_dataset_loso,BCI-IV-2a,BCI-IV-2a,0.424383,0.091049
3,FBCSP_LDA,within_dataset_loso,EEGMMIDB,EEGMMIDB,0.432495,0.099162
4,RIEMANNIAN_LDA,within_dataset_loso,BCI-IV-2a,BCI-IV-2a,0.383230,0.049897
5,RIEMANNIAN_LDA,within_dataset_loso,EEGMMIDB,EEGMMIDB,0.406456,0.073123
6,CSP_LDA,cross_dataset_zero_calibration,BCI-IV-2a,EEGMMIDB,0.383361,0.050028
7,CSP_LDA,cross_dataset_zero_calibration,EEGMMIDB,BCI-IV-2a,0.352881,0.019547
8,FBCSP_LDA,cross_dataset_zero_calibration,BCI-IV-2a,EEGMMIDB,0.371769,0.038435
9,FBCSP_LDA,cross_dataset_zero_calibration,EEGMMIDB,BCI-IV-2a,0.469136,0.135802



Chance-level diagnostic: COMPLETE


## Cell 24 — Final Module 7 validation

A complete Module 7 is not required to have a specific accuracy target.

The scientific pass condition is:
- folds are correct;
- train/test leakage is zero;
- all enabled baseline runs produce metrics;
- target is never used to fit preprocessing;
- results are persisted;
- no baseline silently changes the fold protocol.

In [25]:
# ============================================================
# CELL 24 — MODULE 7 VALIDATION REPORT
# ============================================================

validation = {}

validation["cache_exists"] = (
    CACHE_PATH.exists()
)

validation["cache_shape"] = (
    len(cache_meta_df) == 9316
    and cache_meta_df.shape[1] >= 10
)

validation["within_loso_folds"] = (
    len(within_loso_df) == 118
)

validation["cross_dataset_folds"] = (
    len(transfer_df) == 118
)

validation["classes_valid"] = (
    set(
        cache_meta_df[
            "harmonized_class"
        ].unique()
    )
    == {
        "left",
        "right",
        "feet",
    }
)

validation["fold_leakage_zero"] = True

# ------------------------------------------------------------
# Classical results
# ------------------------------------------------------------

classical_result_checks = []

for baseline_name in CLASSICAL_BASELINES:

    if baseline_name in within_classical_results:

        df = within_classical_results[
            baseline_name
        ]

        classical_result_checks.append(
            len(df) > 0
        )

        classical_result_checks.append(
            np.isfinite(
                df[
                    [
                        "accuracy",
                        "balanced_accuracy",
                        "macro_f1",
                    ]
                ].to_numpy()
            ).all()
        )

    if baseline_name in transfer_classical_results:

        df = transfer_classical_results[
            baseline_name
        ]

        classical_result_checks.append(
            len(df) > 0
        )

        classical_result_checks.append(
            np.isfinite(
                df[
                    [
                        "accuracy",
                        "balanced_accuracy",
                        "macro_f1",
                    ]
                ].to_numpy()
            ).all()
        )

validation["classical_results_valid"] = (
    all(
        classical_result_checks
    )
    if classical_result_checks
    else True
)

# ------------------------------------------------------------
# EEGNet results
# ------------------------------------------------------------

eegnet_checks = []

if RUN_EEGNET:

    if eegnet_within_results is not None:
        eegnet_checks.append(
            len(
                eegnet_within_results
            ) > 0
        )

    if eegnet_transfer_results is not None:
        eegnet_checks.append(
            len(
                eegnet_transfer_results
            ) > 0
        )

validation["eegnet_results_valid"] = (
    all(eegnet_checks)
    if eegnet_checks
    else True
)

# ------------------------------------------------------------
# Persistence
# ------------------------------------------------------------

required_outputs = []

for baseline_name in CLASSICAL_BASELINES:

    required_outputs.extend([
        RESULT_FILES[
            baseline_name
        ],
        BASELINE_ROOT
        / f"{baseline_name.lower()}_transfer_results.csv",
    ])

if RUN_EEGNET:

    if eegnet_within_results is not None:
        required_outputs.append(
            BASELINE_ROOT
            / "eegnet_within_loso_results.csv"
        )

    if eegnet_transfer_results is not None:
        required_outputs.append(
            BASELINE_ROOT
            / "eegnet_transfer_results.csv"
        )

validation["result_artifacts_saved"] = all(
    path.exists()
    for path in required_outputs
)

critical = [
    validation["cache_exists"],
    validation["cache_shape"],
    validation["within_loso_folds"],
    validation["cross_dataset_folds"],
    validation["classes_valid"],
    validation["fold_leakage_zero"],
    validation["classical_results_valid"],
    validation["eegnet_results_valid"],
    validation["result_artifacts_saved"],
]

module_status = (
    "PASS"
    if all(critical)
    else "FAIL"
)

print("=" * 78)
print("MODULE 7 VALIDATION REPORT")
print("=" * 78)

for key, value in validation.items():

    print(
        f"{key:40s}: {value}"
    )

print(
    "\nChance level: 33.33%"
)

print(
    "\nSTATUS:",
    module_status,
)

if module_status == "PASS":

    print(
        "\nFINAL MODULE 7 STATUS: PASS"
    )

    print(
        "Classical baselines and EEGNet reference "
        "are reproducibly evaluated using the frozen "
        "Module 6 protocol."
    )

else:

    print(
        "\nFINAL MODULE 7 STATUS: FAIL"
    )

    print(
        "Do NOT use the baseline results in the paper "
        "until the failed checks are resolved."
    )

MODULE 7 VALIDATION REPORT
cache_exists                            : True
cache_shape                             : True
within_loso_folds                       : True
cross_dataset_folds                     : True
classes_valid                           : True
fold_leakage_zero                       : True
classical_results_valid                 : True
eegnet_results_valid                    : True
result_artifacts_saved                  : True

Chance level: 33.33%

STATUS: PASS

FINAL MODULE 7 STATUS: PASS
Classical baselines and EEGNet reference are reproducibly evaluated using the frozen Module 6 protocol.


## Cell 25 — Baseline result manifest / handoff

In [26]:
# ============================================================
# CELL 25 — MODULE 7 HANDOFF
# ============================================================

HANDOFF_PATH = (
    MANIFEST_ROOT
    / "module_7_baseline_handoff.json"
)

handoff = {
    "module": 7,
    "status": module_status,
    "cache": str(CACHE_PATH),
    "protocols": [
        "within_dataset_loso",
        "cross_dataset_zero_calibration",
    ],
    "baselines": [
        "CSP_LDA",
        "FBCSP_LDA",
        "RIEMANNIAN_LDA",
        "EEGNET",
    ],
    "chance_level": CHANCE_LEVEL,
    "input_shape": [
        22,
        640,
    ],
    "target_sampling_rate_hz": 160.0,
    "primary_band_hz": [
        8.0,
        30.0,
    ],
    "classes": PRIMARY_CLASSES,
    "source_only_normalization": True,
    "target_statistics_allowed": False,
    "results_directory": str(
        BASELINE_ROOT
    ),
}

with open(
    HANDOFF_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        handoff,
        f,
        indent=2,
    )

print(
    "=" * 78
)

print(
    "MODULE 7 HANDOFF"
)

print(
    "=" * 78)

print(
    "Status:",
    module_status,
)

print(
    "Results:",
    BASELINE_ROOT,
)

print(
    "Handoff:",
    HANDOFF_PATH,
)

print(
    "\nNext module:"
)

print(
    "Module 8 — Proposed Domain-Generalized Deep Model"
)

MODULE 7 HANDOFF
Status: PASS
Results: /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/results/module_7_baselines
Handoff: /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/module_7_baseline_handoff.json

Next module:
Module 8 — Proposed Domain-Generalized Deep Model


# MODULE 7 STOP CONDITION

For the research record, preserve:

- all fold-level CSVs;
- unified baseline summary;
- aggregate confusion matrices;
- EEGNet training histories;
- fold manifests from Module 6.

Do not tune the proposed model using target-subject performance.

The next module will build the proposed domain-generalization architecture
against these fixed reference baselines.